# Pinecone 예제 02: Wikipedia RAG

첨부 파일의 `wiki index` 예제를 기준으로 독립 실행 노트북을 구성했습니다.

실습 흐름:
1. `wiki` 인덱스와 `wiki-ns1` 네임스페이스를 사용합니다.
2. 한국어 Wikipedia 데이터 범위를 넓혀 가져옵니다.
3. Wikipedia 문서를 청크로 나누고 `full_text` metadata에 원문 청크를 저장합니다.
4. `PineconeVectorStore.from_existing_index()`로 LangChain 검색기를 만듭니다.
5. 세 질문 모두 Pinecone에 임베딩된 Wikipedia 청크를 검색해 답변합니다.

중요: 직접 작성한 보강 참고 문서는 사용하지 않습니다. 답변 근거는 모두 `wikimedia/wikipedia` 데이터셋에서 가져온 문서입니다.


In [ ]:
import sys
print(sys.executable)


## 1. 패키지 설치

Windows 환경에서 NumPy 2.4.x와 `datasets` 조합이 충돌할 수 있어 `numpy<2.0`으로 고정했습니다. 이 셀 실행 후 이미 NumPy가 로드되어 있었다면 커널을 재시작하세요.


In [ ]:
%pip uninstall -y pinecone-client
%pip install -qU "numpy<2.0" pinecone python-dotenv langchain-openai langchain-text-splitters langchain-pinecone datasets tqdm


## 2. 환경변수와 모델 설정

`WIKI_DATASET_SPLIT`으로 Wikipedia 범위를 조절할 수 있습니다. 기본값은 첨부 파일의 `train[:100]`보다 넓은 `train[:5000]`입니다. 검색 결과가 약하면 `.env` 또는 아래 변수에서 `train[:10000]`, `train[:20000]`처럼 더 늘려 실행하세요.


In [ ]:
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from datasets import load_dataset
from tqdm.auto import tqdm
from uuid import uuid4
import os
import time

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError(".env 파일에 PINECONE_API_KEY가 없습니다.")
if not OPENAI_API_KEY:
    raise ValueError(".env 파일에 OPENAI_API_KEY가 없습니다.")

pc = Pinecone(api_key=PINECONE_API_KEY)

PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

INDEX_NAME = os.getenv("PINECONE_RAG_INDEX", "wiki")
NAMESPACE = os.getenv("PINECONE_RAG_NAMESPACE", "wiki-ns1")

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")

DATASET_NAME = "wikimedia/wikipedia"
DATASET_CONFIG = "20231101.ko"
DATASET_SPLIT = os.getenv("WIKI_DATASET_SPLIT", "train[:5000]")

BATCH_SIZE = 20
FORCE_REINDEX = False


## 3. Pinecone 인덱스 준비

첨부 파일은 `wiki` 인덱스를 바로 생성합니다. 여기서는 이미 존재하는 경우 재사용하고, 차원이 다를 때만 명확히 중단하도록 보완했습니다.


In [ ]:
def list_index_names():
    indexes = pc.list_indexes()
    if hasattr(indexes, "names"):
        return indexes.names()
    return [idx.get("name") if isinstance(idx, dict) else idx.name for idx in indexes]


def get_index_dimension(index_name):
    description = pc.describe_index(index_name)
    if isinstance(description, dict):
        return description.get("dimension")
    return getattr(description, "dimension", None)


def wait_until_ready(index_name, timeout_seconds=120):
    start = time.time()
    while time.time() - start < timeout_seconds:
        description = pc.describe_index(index_name)
        status = description.get("status") if isinstance(description, dict) else getattr(description, "status", {})
        ready = status.get("ready") if isinstance(status, dict) else getattr(status, "ready", False)
        if ready:
            return True
        time.sleep(3)
    raise TimeoutError(f"인덱스 준비 시간이 초과되었습니다: {index_name}")


def ensure_index(index_name, dimension, metric="cosine"):
    indexes = list_index_names()
    print("생성된 인덱스:", indexes)

    if index_name not in indexes:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
        )
        wait_until_ready(index_name)
    else:
        existing_dimension = get_index_dimension(index_name)
        if existing_dimension != dimension:
            raise ValueError(
                f"'{index_name}' 인덱스 차원은 {existing_dimension}입니다. "
                f"이 예제는 {dimension}차원 임베딩이 필요합니다."
            )
    return pc.Index(index_name)

index = ensure_index(INDEX_NAME, EMBEDDING_DIMENSION)
host = pc.describe_index(INDEX_NAME).host
print("host:", host)
print(index.describe_index_stats())

embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL, dimensions=EMBEDDING_DIMENSION)
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)


## 4. 한국어 Wikipedia 데이터 로드와 확인

첨부 파일의 `train[:100]`보다 넓은 범위를 가져옵니다. 아래 키워드가 데이터 범위 안에 있는지도 같이 확인합니다.


In [ ]:
data = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT)
print(data)
print(data[0])
print(data[0]["text"][:1000])
print("첫 문서 길이:", len(data[0]["text"]))

TARGET_KEYWORDS = ["대한민국", "수도", "삼성전자", "한식", "한국 요리", "전통 음식"]
keyword_hits = {keyword: [] for keyword in TARGET_KEYWORDS}

for record in data:
    title = record.get("title", "") or ""
    text = record.get("text", "") or ""
    haystack = f"{title}\n{text[:2000]}"
    for keyword in TARGET_KEYWORDS:
        if keyword in haystack and len(keyword_hits[keyword]) < 5:
            keyword_hits[keyword].append(title)

for keyword, titles in keyword_hits.items():
    print(f"[{keyword}] 발견 문서:", titles)

missing_keywords = [keyword for keyword, titles in keyword_hits.items() if not titles]
if missing_keywords:
    print("주의: 다음 키워드는 현재 DATASET_SPLIT 범위에서 발견되지 않았습니다:", missing_keywords)
    print("검색 품질이 낮으면 WIKI_DATASET_SPLIT을 train[:10000] 또는 train[:20000]처럼 늘려 다시 실행하세요.")


## 5. 텍스트 분할

첨부 파일의 설정과 맞춰 `chunk_size=400`, `chunk_overlap=20`을 사용합니다.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

sample_chunks = splitter.split_text(data[0]["text"])
print("first title:", data[0]["title"])
print("first document chunk count:", len(sample_chunks))
print(sample_chunks[0])


## 6. Wikipedia 청크 업서트

직접 작성한 참고 문서는 넣지 않습니다. `wikimedia/wikipedia` 데이터셋에서 가져온 문서만 청크로 나누고 임베딩합니다.

첨부 파일은 `full_text`를 300자로 잘라 저장했지만, `PineconeVectorStore`의 `text_key`로 쓰려면 원문 청크 전체를 저장하는 편이 검색 결과 확인에 더 좋습니다. 그래서 `full_text`에는 청크 전체를 저장하고, 미리보기용 `preview_text`를 별도로 둡니다.


In [ ]:
def namespace_vector_count(index, namespace):
    stats = index.describe_index_stats()
    namespaces = stats.get("namespaces", {}) if isinstance(stats, dict) else getattr(stats, "namespaces", {})
    namespace_stats = namespaces.get(namespace, {}) if namespaces else {}
    if isinstance(namespace_stats, dict):
        return namespace_stats.get("vector_count", 0)
    return getattr(namespace_stats, "vector_count", 0)


def flush_batch(index, texts, metas):
    if not texts:
        return 0
    ids = [str(uuid4()) for _ in texts]
    embeddings = embedding_model.embed_documents(texts)
    records = [
        {"id": record_id, "values": vector, "metadata": metadata}
        for record_id, vector, metadata in zip(ids, embeddings, metas)
    ]
    index.upsert(vectors=records, namespace=NAMESPACE)
    return len(records)


if FORCE_REINDEX:
    index.delete(delete_all=True, namespace=NAMESPACE)
    time.sleep(3)

existing_count = namespace_vector_count(index, NAMESPACE)
print("기존 벡터 수:", existing_count)

if existing_count == 0 or FORCE_REINDEX:
    texts = []
    metas = []
    total_upserted = 0

    for doc_idx, sample in enumerate(tqdm(data)):
        full_text = sample["text"] or ""
        metadata = {
            "source_type": "wikipedia_chunk",
            "wiki_id": str(sample["id"]),
            "url": sample["url"],
            "title": sample["title"],
            "doc_idx": doc_idx,
        }

        chunks = splitter.split_text(full_text)
        for chunk_idx, chunk in enumerate(chunks):
            clean_chunk = chunk.strip()
            if not clean_chunk:
                continue

            record = {
                "chunk_id": chunk_idx,
                "full_text": clean_chunk,
                "preview_text": clean_chunk[:300],
                **metadata,
            }
            texts.append(clean_chunk)
            metas.append(record)

            if len(texts) >= BATCH_SIZE:
                total_upserted += flush_batch(index, texts, metas)
                texts, metas = [], []
                time.sleep(1)

    total_upserted += flush_batch(index, texts, metas)
    time.sleep(5)
    print("새로 업서트한 Wikipedia 벡터 수:", total_upserted)
else:
    print("기존 벡터가 있어 재색인을 건너뜁니다. FORCE_REINDEX=True로 바꾸면 다시 색인합니다.")

print("현재 벡터 수:", namespace_vector_count(index, NAMESPACE))


## 7. LangChain PineconeVectorStore 검색

첨부 파일과 동일하게 기존 Pinecone 인덱스를 `PineconeVectorStore`로 연결하고, `text_key="full_text"`를 명시합니다.


In [ ]:
vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    text_key="full_text",
    namespace=NAMESPACE,
)

results = vectorstore.similarity_search(
    query="\ub300\ud55c\ubbfc\uad6d\uc758 \uc218\ub3c4\ub294 \uc5b4\ub514\uc778\uac00\uc694?",
    k=3,
    namespace=NAMESPACE,
)

print(f"\uacb0\uacfc \uc218: {len(results)}")
for i, result in enumerate(results, start=1):
    print(f"\n[ {i} ]")
    print(f"page_content: {result.page_content[:100]}")
    print(f"metadata: {result.metadata}")


## 8. RAG 답변 함수

세 질문 모두 Pinecone에 저장된 Wikipedia 청크를 검색한 뒤, 검색 문맥만 근거로 답변합니다. 수도 질문은 답변 형식을 나라 이름 순 표로 강제합니다. 단, 검색 문맥에 충분한 수도 목록 정보가 없으면 모델은 모른다고 답해야 합니다.


In [ ]:
def print_docs(docs):
    for rank, doc in enumerate(docs, start=1):
        print(f"[{rank}] title={doc.metadata.get('title')} chunk={doc.metadata.get('chunk_id')}")
        print(doc.page_content[:250].replace("\n", " "))
        print("url:", doc.metadata.get("url"))
        print()


def build_context(docs, max_chars=3500):
    blocks = []
    total = 0
    for i, doc in enumerate(docs, start=1):
        block = f"[\ubb38\uc11c {i}] \uc81c\ubaa9: {doc.metadata.get('title')}\nURL: {doc.metadata.get('url')}\n\ub0b4\uc6a9: {doc.page_content}"
        if total + len(block) > max_chars:
            break
        blocks.append(block)
        total += len(block)
    return "\n\n".join(blocks)


def answer_with_rag(question, top_k=5):
    docs = vectorstore.similarity_search(question, k=top_k, namespace=NAMESPACE)
    context = build_context(docs)

    if "\uc218\ub3c4" in question or "capital" in question.lower():
        instruction = "\ub2f5\uc744 \ud45c\ub85c \uc791\uc131\ud558\uc138\uc694. \ud45c \uceec\ub7fc\uc740 `\ub098\ub77c`, `\uc218\ub3c4`, `\uadfc\uac70`\uc785\ub2c8\ub2e4. \uc5ec\ub7ec \ub098\ub77c\uac00 \ub098\uc624\uba74 \ub098\ub77c \uc774\ub984\uc21c\uc73c\ub85c \uc815\ub82c\ud558\uc138\uc694."
    else:
        instruction = "\ud575\uc2ec\ub9cc \ud55c\uad6d\uc5b4\ub85c \uac04\uacb0\ud558\uac8c \ub2f5\ud558\uc138\uc694."

    prompt = f"""
\uc544\ub798 \ucc38\uace0 \ubb38\uc11c\ub9cc \uadfc\uac70\ub85c \ud55c\uad6d\uc5b4\ub85c \ub2f5\ud558\uc138\uc694.
\ucc38\uace0 \ubb38\uc11c\uc5d0 \ub2f5\uc774 \uc5c6\uc73c\uba74 \ucd94\uce21\ud558\uc9c0 \ub9d0\uace0 \ubaa8\ub978\ub2e4\uace0 \ub2f5\ud558\uc138\uc694.
{instruction}

[\ucc38\uace0 \ubb38\uc11c]
{context}

[\uc9c8\ubb38]
{question}
""".strip()
    response = llm.invoke(prompt)
    return response.content, docs

question = "\ub300\ud55c\ubbfc\uad6d \uc218\ub3c4\uac00 \uc5b4\ub514\uc778\uc9c0"
answer, answer_docs = answer_with_rag(question)
print(answer)
print("\n\uac80\uc0c9 \uadfc\uac70")
print_docs(answer_docs)


## 9. 여러 질문 검색 확인

첨부 파일의 마지막 검색 테스트처럼 세 질문을 확인합니다. 세 질문 모두 같은 Wikipedia 벡터DB에서 검색합니다.


In [ ]:
queries = [
    "\ub300\ud55c\ubbfc\uad6d\uc758 \uc218\ub3c4\ub294 \uc5b4\ub514\uc778\uac00\uc694?",
    "\uc0bc\uc131\uc804\uc790\ub294 \uc5b4\ub5a4 \ud68c\uc0ac\uc778\uac00\uc694?",
    "\ud55c\uad6d\uc758 \uc804\ud1b5 \uc74c\uc2dd\uc740 \ubb34\uc5c7\uc778\uac00\uc694?",
]

for query in queries:
    print(f"[\uc9c8\ubb38] {query}")
    answer, docs = answer_with_rag(query, top_k=5)
    print(answer)
    print("\n\uac80\uc0c9 \uadfc\uac70")
    print_docs(docs)
    print("-" * 80)


## 10. 최종 검증

아래 셀은 세 질문이 모두 Wikipedia 벡터DB에서 문서를 검색하는지 확인합니다. 답변 정확도는 데이터 범위에 좌우됩니다. 결과가 약하면 `WIKI_DATASET_SPLIT`을 더 크게 늘리고 `FORCE_REINDEX=True`로 다시 색인하세요.


In [ ]:
final_count = namespace_vector_count(index, NAMESPACE)
assert final_count > 0, "RAG \ub124\uc784\uc2a4\ud398\uc774\uc2a4\uc5d0 \ubca1\ud130\uac00 \uc5c6\uc2b5\ub2c8\ub2e4. \uc5c5\uc11c\ud2b8 \uc140\uc744 \uba3c\uc800 \uc2e4\ud589\ud558\uc138\uc694."

validation_queries = [
    "\ub300\ud55c\ubbfc\uad6d\uc758 \uc218\ub3c4\ub294 \uc5b4\ub514\uc778\uac00\uc694?",
    "\uc0bc\uc131\uc804\uc790\ub294 \uc5b4\ub5a4 \ud68c\uc0ac\uc778\uac00\uc694?",
    "\ud55c\uad6d\uc758 \uc804\ud1b5 \uc74c\uc2dd\uc740 \ubb34\uc5c7\uc778\uac00\uc694?",
]

for validation_query in validation_queries:
    validation_answer, validation_docs = answer_with_rag(validation_query, top_k=5)
    assert len(validation_docs) > 0, f"\uac80\uc0c9 \uacb0\uacfc\uac00 \uc5c6\uc2b5\ub2c8\ub2e4: {validation_query}"
    assert all(doc.metadata.get("source_type") == "wikipedia_chunk" for doc in validation_docs), "Wikipedia \uccad\ud06c\uac00 \uc544\ub2cc \ubb38\uc11c\uac00 \uac80\uc0c9\ub418\uc5c8\uc2b5\ub2c8\ub2e4."
    print("\uc9c8\ubb38:", validation_query)
    print("\ub2f5\ubcc0 \ubbf8\ub9ac\ubcf4\uae30:", validation_answer[:300])
    print("\uac80\uc0c9 \ubb38\uc11c \uc81c\ubaa9:", [doc.metadata.get("title") for doc in validation_docs])
    print()

print("\uc608\uc81c 02 \uac80\uc99d \uc644\ub8cc")
print("\uc0c9\uc778 \ubca1\ud130 \uc218:", final_count)
